# Gini y entropía sobre estado civil

Elegimos estado civil porque tiene seis categorías y nos permite comparar ambos índices sobre las mismas frecuencias.

In [1]:
from pathlib import Path
import sys
import os
import tempfile
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "endireh-matplotlib"))
PROYECTO = Path.cwd().resolve()
if PROYECTO.name == "notebooks":
    PROYECTO = PROYECTO.parent
if str(PROYECTO) not in sys.path:
    sys.path.insert(0, str(PROYECTO))
import polars as pl
from IPython.display import display, Image
from config.rutas import ARCHIVO_ENDIREH_PROCESADO, RUTA_FIGURAS
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(40)
pl.Config.set_float_precision(4)
df = pl.read_parquet(ARCHIVO_ENDIREH_PROCESADO)
print(f"Registros: {df.height:,}, columnas: {df.width}, Polars {pl.__version__}")


Registros: 105,715, columnas: 31, Polars 1.44.2


In [2]:
from src.analysis.gini_vs_entropia import comparar_entropia_y_gini_categorias
sintesis = pl.DataFrame([comparar_entropia_y_gini_categorias(df, "estado_civil_id", peso)
                        for peso in [None, "factor_expansion"]])
display(sintesis)

variable,k_categorias,Entropia_Shannon,Gini_Frecuencias,ponderacion
str,i64,f64,f64,str
"""estado_civil_id""",6,2.2176,0.3878,"""sin ponderar"""
"""estado_civil_id""",6,2.2466,0.3717,"""factor_expansion"""


## Diferencia entre Gini y entropía

Para estado civil obtuvimos una entropía ponderada de 2.2466 bits y un Gini de frecuencias de 0.3717. Shannon muestra qué tan repartidas están las respuestas entre las seis categorías. Su máximo es log2(6), aproximadamente 2.5850. El Gini muestra qué tan desiguales son las frecuencias de esas categorías. Los dos usan los mismos datos, pero responden preguntas distintas. Sin ponderar, Shannon es 2.2176. La diferencia se debe al factor de expansión. Estos índices no muestran la relación entre estado civil y violencia. Tampoco hay que confundir el Gini de concentración con Gini-Simpson, que mide diversidad.

## Frecuencias utilizadas

Calculamos una versión sin ponderar, como en el análisis de heterogeneidad, y otra con `factor_expansion`. En cada versión usamos las mismas frecuencias para Gini y Shannon.

In [3]:
display(df.group_by("estado_civil_desc").agg(
    pl.len().alias("frecuencia_muestral"),
    pl.col("factor_expansion").sum().alias("frecuencia_ponderada")
).sort("frecuencia_ponderada"))

estado_civil_desc,frecuencia_muestral,frecuencia_ponderada
str,u32,u32
"""Divorciada""",3172,1343868
"""Separada""",8622,4305266
"""Viuda""",9717,4384227
"""Unión libre""",22206,9056301
"""Soltera""",19798,12831300
"""Casada""",42200,17211997
